# Binning refinement and stability

**Aim:** turn the 10 equal-sized bins from notebook 03 into fewer, more stable bins (coarse classing), compare them with `optbinning`, and check that the bins hold up from train to 2015.

Everything is fitted on 36-month train loans only. The 2015 test year is only used in the stability section.

**Sections**
1. Baseline: the 23 features with decile bins
2. Coarse classing by hand
3. `optbinning` comparison
4. Stability: PSI and WoE by vintage

In [2]:
import sys
sys.path.insert(0, "../src")

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import credit_risk.woe as wo
import credit_risk.features as ft

pd.set_option("display.max_rows", None)
pd.set_option("display.width", 250)

## 1. Baseline: the 23 features with decile bins

The 21 features from `woe.py` plus the 2 kept in notebook 04 (`payment_to_income` and `dti_with_loan`). The new features are built first, then binned the same way as the rest.

In [3]:
train = pd.read_parquet("../data/processed/train.parquet")

# 36-month train loans, with the new ratio features added
train36 = ft.add_features(wo.fit_population(train))

feats = wo.FEATURES + ft.KEPT_FEATURES     # 23 features
rules = wo.fit_woe(train36, features=feats)
wo.iv_summary(rules)

,feature,iv,n_bins,thin_bins
0,sub_grade,0.306364,29,0
1,int_rate,0.283620,10,0
2,grade,0.274793,6,0
3,fico_range_high,0.138752,10,0
4,fico_range_low,0.138752,10,0
5,annual_inc,0.083127,10,0
6,dti_with_loan,0.076781,10,0
7,payment_to_income,0.063292,10,0
8,dti,0.043966,10,0
9,home_ownership,0.034821,3,0


**Baseline result**
- 23 features fitted on 337,996 36-month train loans (the 21 from `woe.py` plus the 2 kept in notebook 04). The IVs of the first 21 are identical to notebook 03.
- The new features: `dti_with_loan` 0.077 and `payment_to_income` 0.063, both with 10 bins and no thin bins.
- Only `addr_state` has a thin bin (under 500 loans), which gets WoE 0.
- These decile-bin IVs are the baseline. Coarse classing can only lower them, so the question is how much.

## 2. Coarse classing by hand

Ten bins is fine classing. Coarse classing merges neighbouring bins whose WoE is close, which leaves fewer bins that each hold more loans.
- Fewer bins means fewer values to estimate, so less noise.
- A scorecard with 4-6 bins per feature is easier to explain than one with 10.
- The cost is some IV, and the question is how much.

**Method:** repeatedly merge the pair of neighbouring bins with the closest WoE, until the target number of bins is reached. The `missing` bin is never merged.

In [4]:
def merge_bins(rule, n_final):
    # merge neighbouring bins until n_final remain, always the pair with the closest WoE
    table, edges = rule["table"], rule["edges"]
    total_good, total_bad = table["non_defaults"].sum(), table["defaults"].sum()
    body = table.drop(index="missing", errors="ignore")      # missing stays on its own

    # each group starts as one original bin; first/last say which bins it covers
    groups = [{"first": int(b), "last": int(b),
               "good": body.loc[b, "non_defaults"], "bad": body.loc[b, "defaults"]}
              for b in sorted(body.index, key=int)]

    def woe(g):
        return np.log((g["good"] / total_good) / (g["bad"] / total_bad))

    while len(groups) > n_final:
        gaps = [abs(woe(groups[i]) - woe(groups[i + 1])) for i in range(len(groups) - 1)]
        i = int(np.argmin(gaps))                              # closest neighbouring pair
        a, b = groups[i], groups[i + 1]
        groups[i:i + 2] = [{"first": a["first"], "last": b["last"],
                            "good": a["good"] + b["good"], "bad": a["bad"] + b["bad"]}]

    rows = []
    for g in groups:
        w = woe(g)
        loans = g["good"] + g["bad"]
        rows.append({"range": f"({edges[g['first']]:.4g}, {edges[g['last'] + 1]:.4g}]",
                     "loans": loans, "default_rate": g["bad"] / loans, "woe": w,
                     "iv": (g["good"] / total_good - g["bad"] / total_bad) * w})
    out = pd.DataFrame(rows)
    iv = out["iv"].sum() + (table.loc["missing", "iv"] if "missing" in table.index else 0)
    return out, iv

In [5]:
col = "revol_util"

# how much IV is lost as bins are merged
for n in [10, 8, 6, 5, 4, 3]:
    _, iv = merge_bins(rules[col], n)
    print(n, "bins  IV:", round(iv, 4))

merged, iv = merge_bins(rules[col], 5)
merged.round(4)

10 bins  IV: 0.0253
8 bins  IV: 0.0253
6 bins  IV: 0.0252
5 bins  IV: 0.0249
4 bins  IV: 0.0236
3 bins  IV: 0.0204


,range,loans,default_rate,woe,iv
0,"(-inf, 21.9]",33858,0.0952,0.3651,0.0117
1,"(21.9, 41.8]",67575,0.1160,0.1439,0.0039
2,"(41.8, 62.8]",101481,0.1324,-0.0075,0.0000
3,"(62.8, 86.4]",101354,0.1434,-0.0995,0.0031
4,"(86.4, inf]",33728,0.1613,-0.2387,0.0062


**Result: `revol_util`**

| Bins | 10 | 8 | 6 | 5 | 4 | 3 |
|---|---|---|---|---|---|---|
| IV | 0.0253 | 0.0253 | 0.0252 | 0.0249 | 0.0236 | 0.0204 |

- Going from 10 bins to 5 loses about 1.6% of the IV. Below 4 bins the cost grows quickly (3 bins lose 19%).
- At 5 bins the default rate rises at every step (9.5% to 16.1%) and WoE falls from +0.37 to −0.24. The middle bin (41.8 to 62.8) has a WoE near 0.
- Decision: 5 bins for `revol_util`.

In [6]:
col = "emp_length"

# the bins before merging, with the default rate added
t = rules[col]["table"].copy()
t["default_rate"] = t["defaults"] / t["loans"]
print(t[["loans", "default_rate", "woe", "iv"]].round(4))

# IV as the numeric bins are merged (the missing bin always stays separate)
for n in [7, 5, 4, 3, 2, 1]:
    _, iv = merge_bins(rules[col], n)
    print(n, "bins + missing  IV:", round(iv, 4))

             loans  default_rate     woe      iv
emp_length                                      
0            50611        0.1363 -0.0409  0.0003
1            31297        0.1320 -0.0034  0.0000
2            27573        0.1318 -0.0022  0.0000
3            21016        0.1295  0.0181  0.0000
4            41886        0.1313  0.0025  0.0000
5            34780        0.1318 -0.0017  0.0000
6           113628        0.1204  0.1016  0.0033
missing      17205        0.1932 -0.4577  0.0126
7 bins + missing  IV: 0.0162
5 bins + missing  IV: 0.0162
4 bins + missing  IV: 0.0162
3 bins + missing  IV: 0.0162
2 bins + missing  IV: 0.016
1 bins + missing  IV: 0.0134


In [7]:
# the 2-bin version: ranges and default rates (the missing bin is not in this table)
merged, iv = merge_bins(rules["emp_length"], 2)
print("IV with missing:", round(iv, 4))
merged.round(4)

IV with missing: 0.016


,range,loans,default_rate,woe,iv
0,"(-inf, 8]",207163,0.1326,-0.0089,0.0000
1,"(8, inf]",113628,0.1204,0.1016,0.0033



**Result: `emp_length`**

| Bins | Loans | Default rate | WoE |
|---|---|---|---|
| 0 to 8 years | 207,163 | 13.26% | −0.01 |
| 9 or more years | 113,628 | 12.04% | +0.10 |
| missing | 17,205 | 19.32% | −0.46 |

- The 6 lower bins of the original 7 are indistinguishable (default rates within 0.7 points, combined IV 0.0003), so they merge into one bin at no cost.
- IV by number of numeric bins, plus missing: 7 to 3 bins 0.0162, 2 bins 0.0160, 1 bin 0.0134.
- Most of the IV is in `missing` (0.0126 of 0.0162).
- Decision: 2 bins plus missing, split at 8 years.


## 3. `optbinning` comparison



`optbinning` finds the bins by optimisation instead of merging by hand. We give it a minimum bin size (5% of loans) and require WoE to move in one direction. First one feature (`revol_util`, where we have hand-merged bins), then all 23.

In [8]:
from optbinning import OptimalBinning

col = "revol_util"
x = train36[col].values            # numpy arrays, to avoid pandas version issues
y = train36["default"].values

# at most 10 bins, each with at least 5% of the loans, WoE rising or falling but not both
optb = OptimalBinning(name=col, dtype="numerical", monotonic_trend="auto_asc_desc",
                      min_bin_size=0.05, max_n_bins=10)
optb.fit(x, y)

table = optb.binning_table.build()
print("IV:", round(table["IV"].iloc[-1], 4))    # the last row is the totals row
table

IV: 0.0263


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 18.35)",26097,0.077211,23667,2430,0.093114,0.389091,0.010112,0.001256
1,"[18.35, 26.45)",19367,0.057299,17346,2021,0.104353,0.26267,0.003586,0.000447
2,"[26.45, 38.15)",40737,0.120525,36020,4717,0.115792,0.145802,0.002427,0.000303
3,"[38.15, 42.85)",19534,0.057794,17133,2401,0.122914,0.078021,0.000342,0.000043
4,"[42.85, 47.85)",22805,0.067471,19864,2941,0.128963,0.02306,0.000036,0.000004
5,"[47.85, 63.15)",75698,0.223961,65558,10140,0.133953,-0.020652,0.000096,0.000012
6,"[63.15, 73.25)",48035,0.142117,41266,6769,0.140918,-0.079414,0.000923,0.000115
7,"[73.25, 86.35)",51399,0.152070,43891,7508,0.146073,-0.121359,0.002342,0.000293
8,"[86.35, 91.95)",16987,0.050258,14296,2691,0.158415,-0.217033,0.002562,0.000320
9,"[91.95, inf)",17077,0.050524,14268,2809,0.164490,-0.261909,0.003812,0.000475


In [9]:
# same fit, but at most 5 bins, to compare with our 5-bin hand merge (IV 0.0249)
optb5 = OptimalBinning(name=col, dtype="numerical", monotonic_trend="auto_asc_desc",
                       min_bin_size=0.05, max_n_bins=5)
optb5.fit(x, y)
t5 = optb5.binning_table.build()
print("IV:", round(t5["IV"].iloc[-1], 4))
t5

IV: 0.0254


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 26.45)",45464,0.134510,41013,4451,0.097902,0.333661,0.013227,0.001646
1,"[26.45, 42.85)",60271,0.178319,53153,7118,0.118100,0.123448,0.002596,0.000324
2,"[42.85, 63.15)",98503,0.291432,85422,13081,0.132798,-0.010657,0.000033,0.000004
3,"[63.15, 86.35)",99434,0.294187,85157,14277,0.143583,-0.101253,0.003130,0.000391
4,"[86.35, inf)",34064,0.100782,28564,5500,0.161461,-0.239701,0.006318,0.000788
5,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
6,Missing,260,0.000769,215,45,0.173077,-0.323124,0.000090,0.000011
Totals,,337996,1.000000,293524,44472,0.131576,,0.025395,0.003164


**Result: `optbinning` vs our bins for `revol_util`**

| Version | Bins | Cut points | IV |
|---|---|---|---|
| Deciles | 10 | decile edges | 0.0253 |
| Hand-merged | 5 | 21.9, 41.8, 62.8, 86.4 | 0.0249 |
| `optbinning` | 5 | 26.45, 42.85, 63.15, 86.35 | 0.0254 |
| `optbinning` | 10 | finer at both ends | 0.0263 |

- Three of four cut points agree within about 1 point, so our hand merge picked sensible bins.
- Our merge can only join existing decile bins, while `optbinning` can place cuts anywhere. That explains its extra 0.0005 of IV.
- Default rate rises and WoE falls at every step in both versions.
- `optbinning` maximises IV on train, so its higher IV is expected by construction. The stability section tests whether the bins hold on 2015.
- `optbinning` keeps the 260 missing loans as their own bin. `woe.py` fills a missing group under 500 loans with the median.

In [10]:
from optbinning import OptimalBinning

base = wo.iv_summary(rules).set_index("feature")      # decile-bin baseline from cell 4
cats = ["grade", "sub_grade", "home_ownership", "purpose", "addr_state"]
y = train36["default"].values

opt, rows = {}, []                                    # fitted binnings are kept in opt for later
for col in feats:
    is_cat = col in cats
    x = train36[col].to_numpy(dtype=object) if is_cat else train36[col].to_numpy()
    kwargs = {} if is_cat else {"monotonic_trend": "auto_asc_desc"}   # categories are ordered by default rate already
    try:
        ob = OptimalBinning(name=col, dtype="categorical" if is_cat else "numerical",
                            min_bin_size=0.05, max_n_bins=10, **kwargs)
        ob.fit(x, y)
        t = ob.binning_table.build()
        opt[col] = ob
        rows.append({"feature": col, "iv_baseline": base.loc[col, "iv"], "bins_baseline": base.loc[col, "n_bins"],
                     "iv_optbinning": t["IV"].iloc[-1], "bins_optbinning": len(t) - 3})   # minus Special, Missing, Totals
    except Exception as e:
        rows.append({"feature": col, "iv_baseline": base.loc[col, "iv"], "error": str(e)[:80]})

cmp = pd.DataFrame(rows).set_index("feature")
cmp["iv_change_%"] = (100 * (cmp["iv_optbinning"] / cmp["iv_baseline"] - 1)).round(1)
cmp.sort_values("iv_baseline", ascending=False).round(4)

,iv_baseline,bins_baseline,iv_optbinning,bins_optbinning,iv_change_%
feature,,,,,
sub_grade,0.3064,29,0.2992,10,-2.3
int_rate,0.2836,10,0.2889,10,1.9
grade,0.2748,6,0.2706,4,-1.5
fico_range_high,0.1388,10,0.1422,10,2.5
fico_range_low,0.1388,10,0.1422,10,2.5
annual_inc,0.0831,10,0.0848,10,2.0
dti_with_loan,0.0768,10,0.0795,10,3.6
payment_to_income,0.0633,10,0.0654,10,3.3
dti,0.0440,10,0.0456,10,3.7


**Result: `optbinning` on all 23 features (at most 10 bins, 5% minimum bin size)**
- The strong numeric features change little: IV is 1.9% to 4.0% higher for `int_rate`, `fico_range_low/high`, `annual_inc`, `dti`, `dti_with_loan`, `payment_to_income` and `revol_util`. The optimiser fits train, so some gain is expected by construction.
- `sub_grade` goes from 29 to 10 bins and loses 2.3% of IV. The top three keep their order (`sub_grade` 0.299, `int_rate` 0.289, `grade` 0.271).
- `inq_last_6mths` gains 35% (0.0245 to 0.0332). Our deciles could not split the tied loans at 0 and 1.
- `loan_amnt` (+31%) and `installment` (−43%) move a lot in relative terms but only by 0.0017 each in IV. `installment` zigzags, so forcing WoE to move in one direction costs it signal.
- The categoricals all lost IV: `purpose` −11.8% (5 to 4 bins), `addr_state` −6.3% (47 to 10), `grade` −1.5% (6 to 4). The 5% minimum bin size forces small, risky groups to merge.

In [11]:
# numeric features again with at most 6 bins
nums = [c for c in feats if c not in cats]

opt6, rows6 = {}, []
for col in nums:
    ob = OptimalBinning(name=col, dtype="numerical", monotonic_trend="auto_asc_desc",
                        min_bin_size=0.05, max_n_bins=6)
    ob.fit(train36[col].to_numpy(), y)
    t = ob.binning_table.build()
    opt6[col] = ob
    rows6.append({"feature": col, "iv_10_bins": cmp.loc[col, "iv_optbinning"],
                  "iv_6_bins": t["IV"].iloc[-1], "bins_6": len(t) - 3})

c6 = pd.DataFrame(rows6).set_index("feature")
c6["iv_change_%"] = (100 * (c6["iv_6_bins"] / c6["iv_10_bins"] - 1)).round(1)
c6.sort_values("iv_10_bins", ascending=False).round(4)

,iv_10_bins,iv_6_bins,bins_6,iv_change_%
feature,,,,
int_rate,0.2889,0.2827,6,-2.2
fico_range_low,0.1422,0.1396,6,-1.8
fico_range_high,0.1422,0.1396,6,-1.8
annual_inc,0.0848,0.0828,6,-2.4
dti_with_loan,0.0795,0.0778,6,-2.1
payment_to_income,0.0654,0.0645,6,-1.3
dti,0.0456,0.0447,6,-2.0
inq_last_6mths,0.0332,0.0332,4,0.0
revol_util,0.0263,0.0259,6,-1.8


In [12]:
# grade and purpose with a 2% minimum bin size, to see if the risky tail can stay separate
for col in ["grade", "purpose"]:
    ob = OptimalBinning(name=col, dtype="categorical", min_bin_size=0.02, max_n_bins=10)
    ob.fit(train36[col].to_numpy(dtype=object), y)
    t = ob.binning_table.build()
    print(col, "IV:", round(t["IV"].iloc[-1], 4), " bins:", len(t) - 3)
    display(t[["Bin", "Count", "Event rate", "WoE"]].round(4))

grade IV: 0.2706  bins: 4


,Bin,Count,Event rate,WoE
0,[A],72888,0.0553,0.950918
1,[B],120242,0.1069,0.235694
2,[C],84979,0.1656,-0.270088
3,"[D, E, F, G]",59887,0.2256,-0.654013
4,Special,0,0.0000,0.0
5,Missing,0,0.0000,0.0
Totals,,337996,0.1316,


purpose IV: 0.0231  bins: 4


,Bin,Count,Event rate,WoE
0,"[car, major_purchase, credit_card, wedding]",93556,0.1104,0.19919
1,[home_improvement],18668,0.1153,0.150307
2,"[debt_consolidation, vacation, house]",194673,0.1367,-0.044527
3,"[other, medical, renewable_energy, moving, edu...",31099,0.1725,-0.319408
4,Special,0,0.0000,0.0
5,Missing,0,0.0000,0.0
Totals,,337996,0.1316,


In [13]:
opt["inq_last_6mths"].binning_table.build()[["Bin", "Count", "Event rate", "WoE", "IV"]].round(4)

,Bin,Count,Event rate,WoE,IV
0,"(-inf, 0.50)",175845,0.1141,0.162199,0.0129
1,"[0.50, 1.50)",94485,0.1382,-0.057149,0.0009
2,"[1.50, 2.50)",41556,0.1589,-0.220619,0.0065
3,"[2.50, inf)",26081,0.1816,-0.381475,0.0129
4,Special,0,0.0000,0.0,0.0000
5,Missing,29,0.1034,0.272385,0.0000
Totals,,337996,0.1316,,0.0332


In [14]:
# cut points from the 6-bin fits, as open-ended edges
final_edges = {}
for col in nums:
    splits = np.sort(opt6[col].splits)
    final_edges[col] = np.concatenate([[-np.inf], splits, [np.inf]])

pd.Series({c: np.round(e[1:-1], 2).tolist() for c, e in final_edges.items()})

loan_amnt                                             [11987.5, 19762.5]
int_rate                               [7.64, 9.74, 12.14, 14.48, 17.56]
installment                                             [364.76, 607.44]
emp_length                                          [0.5, 1.5, 3.5, 9.5]
annual_inc                [37086.5, 43317.5, 62010.8, 75000.5, 100131.0]
dti                                    [10.7, 15.57, 19.06, 25.39, 29.2]
credit_history_years                   [6.29, 8.04, 11.71, 15.59, 18.76]
fico_range_low                       [672.5, 687.5, 707.5, 727.5, 757.5]
fico_range_high                      [676.5, 691.5, 711.5, 731.5, 761.5]
inq_last_6mths                                           [0.5, 1.5, 2.5]
mths_since_last_delinq                                       [9.5, 21.5]
open_acc                                               [8.5, 13.5, 19.5]
revol_bal                   [7802.5, 12870.5, 20332.5, 27582.5, 36105.5]
revol_util                           [18.35, 26.45,

## 4. Stability: PSI

Do 2015 loans fall into the bins in the same proportions as train loans? PSI compares the share of loans in each bin. Under 0.1 is stable, 0.1 to 0.25 a moderate shift, above 0.25 a big shift. Only feature values are used, no defaults.

In [15]:
# 2015 loans, with the same ratio features added
test = ft.add_features(pd.read_parquet("../data/processed/test.parquet"))

def bin_share(x, edges):
    # share of loans in each bin; missing values get their own bin
    b = pd.cut(x, edges, labels=False, include_lowest=True).astype("Int64").astype(str)
    return b.where(x.notna(), "missing").value_counts(normalize=True)

def psi(a, b, eps=1e-4):
    # PSI between two bin-share Series (a = train, b = 2015)
    idx = a.index.union(b.index)
    a, b = a.reindex(idx, fill_value=0) + eps, b.reindex(idx, fill_value=0) + eps
    return float(((b - a) * np.log(b / a)).sum())

rows = []
for col in feats:
    if col in final_edges:          # numeric: the new optbinning cut points
        tr, te = bin_share(train36[col], final_edges[col]), bin_share(test[col], final_edges[col])
    else:                           # categories: the groups stored in woe.py
        tr = wo._assign_bins(train36[col], rules[col]).value_counts(normalize=True)
        te = wo._assign_bins(test[col], rules[col]).value_counts(normalize=True)
    rows.append({"feature": col, "psi": psi(tr, te)})

psi_table = pd.DataFrame(rows).set_index("feature").sort_values("psi", ascending=False)
psi_table["status"] = pd.cut(psi_table["psi"], [-1, 0.1, 0.25, 99], labels=["stable", "moderate", "unstable"])
psi_table.round(4)

,psi,status
feature,,
int_rate,0.0940,stable
inq_last_6mths,0.0490,stable
dti,0.0464,stable
dti_with_loan,0.0285,stable
sub_grade,0.0240,stable
addr_state,0.0227,stable
pub_rec_bankruptcies,0.0193,stable
credit_history_years,0.0192,stable
revol_util,0.0160,stable


In [16]:
# int_rate: share of loans per bin, train vs 2015
tr = bin_share(train36["int_rate"], final_edges["int_rate"])
te = bin_share(test["int_rate"], final_edges["int_rate"])
pd.DataFrame({"train": tr, "2015": te}).sort_index().round(3)

,train,2015
int_rate,,
0,0.109,0.176
1,0.151,0.199
2,0.206,0.210
3,0.239,0.225
4,0.174,0.120
5,0.120,0.069


In [17]:
share = pd.DataFrame({"train": tr, "2015": te}).sort_index()
share["diff"] = share["2015"] - share["train"]
share["psi_part"] = share["diff"] * np.log(share["2015"] / share["train"])
print("PSI:", round(share["psi_part"].sum(), 3))
share.round(3)

PSI: 0.094


,train,2015,diff,psi_part
int_rate,,,,
0,0.109,0.176,0.067,0.032
1,0.151,0.199,0.048,0.013
2,0.206,0.210,0.003,0.000
3,0.239,0.225,-0.014,0.001
4,0.174,0.120,-0.054,0.020
5,0.120,0.069,-0.051,0.028


In [22]:
d = train36.copy()
d["bin"] = pd.cut(d["int_rate"], final_edges["int_rate"], labels=False, include_lowest=True)
rate = d.pivot_table(index="bin", columns="issue_year", values="default", aggfunc="mean")
rate.round(3)

issue_year,2007,2008,2009,2010,2011,2012,2013,2014
bin,,,,,,,,
0,0.062,0.036,0.030,0.043,0.053,0.057,0.030,0.039
1,0.118,0.087,0.071,0.057,0.089,0.086,0.062,0.075
2,0.236,0.198,0.115,0.095,0.106,0.115,0.090,0.106
3,0.323,0.250,0.152,0.139,0.134,0.147,0.122,0.151
4,0.436,0.268,0.189,0.184,0.184,0.185,0.165,0.201
5,0.444,0.338,0.294,0.279,0.220,0.214,0.213,0.253


In [23]:
import json

inner = {c: [float(v) for v in e[1:-1]] for c, e in final_edges.items()}
with open("../src/credit_risk/bins.json", "w") as f:
    json.dump(inner, f, indent=1)
print(len(inner), "features saved; revol_util cuts:", inner["revol_util"])

18 features saved; revol_util cuts: [18.34999942779541, 26.449999809265137, 42.85000038146973, 63.14999961853027, 86.35000228881836]


In [24]:
import credit_risk.binning as bn

rule = bn.numeric_rule(train36["revol_util"], bn.BINS["revol_util"])
print(rule["edges"])
print(rule["fill"])

[       -inf 18.34999943 26.44999981 42.85000038 63.14999962 86.35000229
         inf]
56.1


In [25]:
rows = []
for col in bn.BINS:
    rule = bn.numeric_rule(train36[col], bn.BINS[col])
    labels = wo._assign_bins(train36[col], rule)
    table = wo._woe_table(labels, train36["default"], wo.MIN_LOANS)
    rows.append({"feature": col, "iv_binning_py": table["iv"].sum()})

chk = pd.DataFrame(rows).set_index("feature")
chk["iv_optbinning"] = c6["iv_6_bins"]
chk["diff"] = chk["iv_binning_py"] - chk["iv_optbinning"]
chk.round(4)

,iv_binning_py,iv_optbinning,diff
feature,,,
loan_amnt,0.0073,0.0073,-0.0000
int_rate,0.2827,0.2827,0.0000
installment,0.0022,0.0022,0.0000
emp_length,0.0166,0.0166,0.0000
annual_inc,0.0828,0.0828,-0.0000
dti,0.0447,0.0447,0.0000
credit_history_years,0.0196,0.0196,-0.0000
fico_range_low,0.1396,0.1396,0.0000
fico_range_high,0.1396,0.1396,0.0000


**Check: `binning.py` against `optbinning`**
All 18 numeric features match the `optbinning` IVs to 4 decimals. The largest gap is `revol_util` (−0.0001), where `binning.py` fills the 260 missing loans with the median and `optbinning` gives them their own bin.

In [27]:
rules2 = bn.fit_final(train36)
wo.iv_summary(rules2)

,feature,iv,n_bins,thin_bins
0,sub_grade,0.306364,29,0
1,int_rate,0.282714,6,0
2,grade,0.274793,6,0
3,fico_range_high,0.139632,6,0
4,fico_range_low,0.139632,6,0
5,annual_inc,0.082791,6,0
6,dti_with_loan,0.077844,6,0
7,payment_to_income,0.064515,6,0
8,dti,0.044671,6,0
9,home_ownership,0.034821,3,0


**Result: `binning.py` final rules (23 features, 36-month train)**
- The 18 numeric features use the saved `optbinning` cut points (at most 6 bins). The 5 categories use the `woe.py` groups.
- The IVs match the earlier checks: numeric ones match `optbinning` to 4 decimals, and categories match the notebook 03 baseline (`sub_grade` 0.306, `grade` 0.275, `home_ownership` 0.035, `purpose` 0.026, `addr_state` 0.015).
- Only `addr_state` has a thin bin (under 500 loans, WoE 0).
- IV ranking: `sub_grade` 0.306, `int_rate` 0.283, `grade` 0.275, the FICO pair 0.140, `annual_inc` 0.083, `dti_with_loan` 0.078, `payment_to_income` 0.065, `dti` 0.045.

In [28]:
test = ft.add_features(pd.read_parquet("../data/processed/test.parquet"))

train_woe = wo.apply_woe(train36, rules2)
test_woe = wo.apply_woe(test, rules2)

print(train_woe.shape, test_woe.shape)
print((test_woe == 0).mean().round(4).sort_values(ascending=False).head(5))

(337996, 23) (283026, 23)
addr_state              0.0049
loan_amnt               0.0000
fico_range_low          0.0000
payment_to_income       0.0000
pub_rec_bankruptcies    0.0000
dtype: float64


In [29]:
cols = ["id", "issue_year", "default"]
pd.concat([train36[cols], train_woe], axis=1).to_parquet("../data/processed/train_woe_v2.parquet")
pd.concat([test[cols], test_woe], axis=1).to_parquet("../data/processed/test_woe_v2.parquet")

### Decision: final bins

- **Numeric features (18):** `optbinning` cut points, at most 6 bins. Saved in `bins.json`.
- **Categories (5):** our groups from `woe.py`. `optbinning` did worse on them.
- **Missing values:** filled with the median if fewer than 500 loans, otherwise their own bin.
- **Result:** 6 bins lose only about 2% of IV. PSI is under 0.1 for all 23 features.
- **Caveat:** default rates inside bins drift between years, so expect under-prediction on 2015.